# 2 · Compare and decide

**What would make the repair acceptable?** Give the app the current policy document. Keep the cases and checks fixed.


In [ ]:
import importlib
import os, sys, json
from pathlib import Path

# Find the checkout in local Jupyter or a Databricks Git folder.
start = Path(os.environ.get("WORKSHOP_PROJECT_ROOT", Path.cwd())).resolve()
ROOT = next((p for p in (start, *start.parents) if (p / "west_workshop" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set WORKSHOP_PROJECT_ROOT to the workshop checkout, then restart this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
importlib.invalidate_caches()
os.chdir(ROOT)
MODE = os.environ.get("WORKSHOP_MODE", "saved")  # Explicit opt-in before any model calls.


In [ ]:
from IPython.display import display, HTML
from west_workshop.repair_workshop import notebook_session
from west_workshop.mlflow_view import (
    session_header, controls_card, policy_card, inspection_card,
    finding_cases, paired_summary, review_card, decision_card, statistics_card,
)

session = notebook_session(MODE)
display(HTML(session_header(session)))


## Test the judge

Can the judge tell the customer's instructions apart from the assistant's claim that a refund happened?


In [ ]:
controls = None
if not session.has_execution_failure():
    controls = session.qualify_judge()
display(HTML(controls_card(controls)))


## Evaluate the repair

Use the same MLflow call for both versions. Failed controls or an execution error stop further model calls.


In [ ]:
if MODE == "live" and controls and controls["passed"] and not session.has_execution_failure():
    import mlflow
    for variant in ("deployed", "repair"):
        if session.needs(variant):
            with session.evaluation(variant) as run:
                result = mlflow.genai.evaluate(
                    data=run.data, predict_fn=run.predict_fn, scorers=run.scorers,
                )
                session.record_evaluation(variant, result, run)
            if session.has_execution_failure():
                break

review_path = os.environ.get("WORKSHOP_REVIEW_JSON")
reviewed = json.loads(Path(review_path).read_text(encoding="utf-8")) if review_path else None
comparison = session.policy_inputs(reviewed)


## Code view 3: apply the acceptance rules

MLflow records the results. These rules decide whether the repair passes.


In [ ]:
from west_workshop.repair_policy import DEFAULT_POLICY, evaluate_repair

policy_result = evaluate_repair(**comparison, policy=DEFAULT_POLICY)


In [ ]:
report = session.finish(comparison["review"])
assert report["policy_result"] == policy_result
display(HTML(policy_card(DEFAULT_POLICY)))


## Inspect in MLflow: repaired answer → disputed grade

Check the repaired day-45 source and answer. Then inspect any case flagged by the review.
MLflow records the trace and scores. The review records any unresolved issue.


In [ ]:
display(HTML(inspection_card(session, "repair", ["day_45_opening", *finding_cases(report)])))


## Compare each case and read the review

Compare the same case before and after. A 0-to-0 score is a tie, even on a protected case.
A material finding is an issue that can affect acceptance. A completed review can leave one unresolved.


In [ ]:
display(HTML(paired_summary(report)))


In [ ]:
display(HTML(review_card(report)))


## Vote before revealing the decision

**Pause for 20 seconds.** PASS, BLOCK, REVIEW REQUIRED, or INVALID EVIDENCE?
Name the evidence that would change your decision.


In [ ]:
display(HTML(decision_card(report)))


## Turn the finding into a future test

Preserve the original answer and score. Have a human approve the new case label.
Test a changed judge on controls. Then use that version to grade all the same saved answers again.
Keep the original scores.

The README shows where to change the app, cases, checks and acceptance rules. Record who owns the next decision and the evidence they need.


In [ ]:
display(HTML(statistics_card(report)))
